# Нейронные сети и глубокое обучение, МНАД ВШЭ

## Домашнее задание 4. Трансформеры. 

### Общая информация

### Оценивание и штрафы

Максимально допустимая оценка за работу без бонусов — 10 баллов. Сдавать задание после указанного срока жесткого дедлайна нельзя.

Сдача работы после мягкого дедлайна штрафуется ступенчато, -1 балл в сутки. Один раз за модуль студентам предоставляется возможность использовать отсрочку и сдать в жесткий дедлайн без штрафа.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

Использование генеративных моделей допустимо на следующих условиях:
- Количество кода, написанное генеративными моделями, не превышает 30%
- Указана модель, использованная для генерации, а также промпт
- В конце работы необходимо описать свой опыт использования генеративного ИИ для решения данного домашнего задания. Укажите как часто Вам приходилось исправлять код своими руками или просить модель что-то исправить. Было ли это быстрее, чем написать код самим? 

В случае невыполнения этих требований работа не оценивается и оценка за неё не превышает 0 баллов.

### О задании

В этой домашней работе вам предстоит добавить к BERT'у декодерную часть и решить задачу написания tl;dr для текстов новостей на русском языке.

Дополнительно к этому на отличную оценку потребуется реализовать менее жадную стратегию выбора следующего токена для генерации.

In [1]:
!pip install transformers datasets evaluate


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, BertModel, BertTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("устройство:", device)

## Подготовка данных (0.5 балла)

Мы воспользуемся датасетом с 🤗 Ильи Гусева "gazeta". Он представляет собой пары (полный текст новости -- его саммари). 

Более подробно про датасет можно прочитать [здесь](https://huggingface.co/datasets/IlyaGusev/gazeta)



In [3]:
# Загрузим данные с попощью datasets
# Вы вольны взять меньше или больше данных, но что-то около адекватное получается обычно только на >=10%

from datasets import load_dataset

dataset = load_dataset("IlyaGusev/gazeta", split="train[:10%]")
val_dataset = load_dataset("IlyaGusev/gazeta", split="validation")

# смотрим на один пример и имена полей
print(dataset[0].keys())
print("\n--- текст статьи (начало) ---")
print(dataset[0]["text"][:300])
print("\n--- краткое содержание ---")
print(dataset[0]["summary"])

README.md: 0.00B [00:00, ?B/s]

default/train/0000.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

default/train/0001.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/27.8M [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60964 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6369 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6793 [00:00<?, ? examples/s]

dict_keys(['text', 'summary', 'title', 'date', 'url'])

--- текст статьи (начало) ---
Сегодня транспортный налог начисляется в зависимости от мощности автомобиля, причем цена для «сильных» машин выше, чем для малолитражек. Также ставку налога могут корректировать региональные власти: согласно Налоговому кодексу, базовый тариф, установленный правительством, может быть уменьшен в пять 

--- краткое содержание ---
С 2011 года правительство отменяет самый раздражающий граждан налог – транспортный. Но поборы автомобилистов не прекратятся – налоги завуалируют в бензиновые акцизы и платные дороги, а цены на товары подскочат. Зато теперь собираемые деньги обещают пустить только на строительство и содержание дорог.


Вы должны помнить, что тексты перед подачей в модель необходимо **токенизировать**.

Добавьте паддинг до `max_length=512` для обучающих данных, а также до `max_length=128` для меток.

Используйте обрезку текстов, длина которых в токенах превышает `max_length`

In [4]:
# Подготовим данные для модели Bert

model_name = "deepvk/bert-base-uncased"  # Указание модели BERT

tokenizer = AutoTokenizer.from_pretrained(model_name)


def preprocess(examples, use_padding=True):
    # токенизируем текст статьи - получаем input_ids и attention_mask
    model_inputs = tokenizer(
        examples["text"],
        padding="max_length" if use_padding else False,
        truncation=True,
        max_length=512,
    )

    # токенизируем краткое содержание - это будут метки для обучения
    labels = tokenizer(
        examples["summary"],
        padding="max_length" if use_padding else False,
        truncation=True,
        max_length=128,
    )

    # меняем паддинг-токены в метках на -100 - они не будут учитываться в ошибке
    label_ids = [
        l if l != tokenizer.pad_token_id else -100
        for l in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids

    return model_inputs

tokenizer_config.json:   0%|          | 0.00/332 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [5]:
tokenized_dataset = dataset.map(preprocess, batched=False)
tokenized_val_dataset = val_dataset.map(preprocess, batched=False)

# оставляем только те колонки, которые нужны pytorch - строковые поля сломают загрузчик
cols = ["input_ids", "attention_mask", "labels"]
tokenized_dataset.set_format("torch", columns=cols)
tokenized_val_dataset.set_format("torch", columns=cols)

Map:   0%|          | 0/6096 [00:00<?, ? examples/s]

Map:   0%|          | 0/6369 [00:00<?, ? examples/s]

In [6]:
from torch.utils.data import DataLoader

# shuffle=True для обучения - каждую эпоху примеры идут в разном порядке
train_dataloader = DataLoader(tokenized_dataset, batch_size=8, shuffle=True)
eval_dataloader = DataLoader(tokenized_val_dataset, batch_size=8, shuffle=False)

In [7]:
# проверка - берём один батч и смотрим на его содержимое
batch = next(iter(train_dataloader))

print("ключи батча:", list(batch.keys()))
print("форма input_ids:     ", batch["input_ids"].shape)
print("форма attention_mask:", batch["attention_mask"].shape)
print("форма labels:        ", batch["labels"].shape)

ключи батча: ['input_ids', 'attention_mask', 'labels']
форма input_ids:      torch.Size([8, 512])
форма attention_mask: torch.Size([8, 512])
форма labels:         torch.Size([8, 128])


In [8]:
print(type(tokenized_dataset))
print(type(tokenized_val_dataset))
print(tokenized_dataset[0].keys())
print(tokenized_val_dataset[0].keys())

<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>
dict_keys(['input_ids', 'attention_mask', 'labels'])
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [10]:
len(train_dataloader), len(eval_dataloader)

(762, 797)

## Реализация Decoder-cети (3 балла)

В данном разделе вам необходимо **реализовать собственный декодер для генерации текста**.

Можете вдохновляться кодом с семинара. В инициализации весов стоит (но необязательно) вспомнить нюансы.

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer

# Класс модели для суммаризации на основе BERT с кастомным декодером


class BertSummarizer(nn.Module):
    def __init__(
        self,
        bert_model_name="bert-base-uncased",
        hidden_size=768,
        num_decoder_layers=3,
        num_heads=8,
        dropout=0.1,
    ):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.hidden_size = hidden_size

        # Эмбеддинги для токенов на входе в декодер
        self.embedding = nn.Embedding(self.bert.config.vocab_size, hidden_size)

        # декодер - принимает эмбеддинги summary и выход берта, предсказывает следующий токен
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dropout=dropout,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)

        # линейный слой - переводит выход декодера обратно в размер словаря
        self.fc_out = nn.Linear(hidden_size, self.bert.config.vocab_size)

        # вероятности по каждому токену словаря
        self.softmax = nn.LogSoftmax(dim=-1)

        # инициализация весов - помогает модели стабильнее обучаться с нуля
        nn.init.xavier_uniform_(self.fc_out.weight)

    # Функция для создания маски для предотвращения заглядывания вперед в декодере

    def generate_square_subsequent_mask(self, T):
        # верхний треугольник -inf - токен не может смотреть на будущие позиции
        return torch.triu(torch.full((T, T), float("-inf")), diagonal=1)

    def forward(self, input_ids, attention_mask, decoder_input_ids):
        encoder_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        memory = (
            encoder_outputs.last_hidden_state
        )  # Выходы BERT для использования в декодере

        # Эмбеддинги для входных токенов декодера
        embedded = self.embedding(decoder_input_ids)

        # декодер ждёт (seq_len, batch, hidden) - переставляем оси
        embedded = embedded.transpose(0, 1)
        memory = memory.transpose(0, 1)

        T = embedded.size(0)
        mask = self.generate_square_subsequent_mask(T).to(input_ids.device)

        decoder_output = self.decoder(tgt=embedded, memory=memory, tgt_mask=mask)

        # возвращаем обратно в (batch, seq_len, vocab_size)
        output = self.fc_out(decoder_output.transpose(0, 1))

        return self.softmax(output)

    def generate(self, input_ids, attention_mask, tokenizer, max_len=50):
        encoder_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        memory = encoder_outputs.last_hidden_state
        batch_size = input_ids.size(0)

        # Начинаем с токена [CLS] или [BOS] (начало последовательности)
        decoder_input_ids = torch.full(
            (batch_size, 1), tokenizer.cls_token_id, dtype=torch.long
        ).to(input_ids.device)
        memory = memory.transpose(0, 1)
        generated_tokens = []

        for _ in range(max_len):
            embedded = self.embedding(decoder_input_ids).transpose(0, 1)

            # Генерация маски для предотвращения заглядывания вперед
            decoder_attention_mask = self.generate_square_subsequent_mask(
                embedded.size(0)
            ).to(input_ids.device)
            decoder_output = self.decoder(
                tgt=embedded, memory=memory, tgt_mask=decoder_attention_mask
            )

            output = self.fc_out(decoder_output.transpose(0, 1))

            # Получаем индекс токена с наибольшей вероятностью.
            # Помните, если EOS предсказан, прекращаем генерацию

            # берём предсказание только на последней позиции - она отвечает за следующий токен
            next_token = output[:, -1, :].argmax(dim=-1, keepdim=True)

            if next_token.item() == tokenizer.eos_token_id:
                break

            # добавляем новый токен в последовательность - на следующем шаге декодер его увидит
            decoder_input_ids = torch.cat([decoder_input_ids, next_token], dim=1)

        generated_sequence = tokenizer.decode(
            decoder_input_ids.squeeze().tolist(), skip_special_tokens=True
        )

        return generated_sequence

In [ ]:
# проверка маски на маленьком примере
import torch
tmp = torch.triu(torch.full((5, 5), float("-inf")), diagonal=1)
print(tmp)

In [ ]:
# проверка forward на одном батче
model_test = BertSummarizer(bert_model_name=model_name)
model_test.eval()

batch = next(iter(train_dataloader))
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
# labels используем как decoder_input_ids - заменяем -100 на 0, чтобы не было ошибки
decoder_input_ids = batch["labels"].clone()
decoder_input_ids[decoder_input_ids == -100] = 0

with torch.no_grad():
    out = model_test(input_ids, attention_mask, decoder_input_ids)

print("форма выхода:", out.shape)
# ожидаем (batch_size, seq_len_labels, vocab_size) = (8, 128, vocab_size)

In [ ]:
# проверка generate - модель ещё не обучена, текст будет бессмысленным, но тип должен быть str
model_test.eval()

sample = next(iter(eval_dataloader))
with torch.no_grad():
    result = model_test.generate(
        sample["input_ids"][:1],
        sample["attention_mask"][:1],
        tokenizer,
    )

print(type(result))   # должен быть <class 'str'>
print(result[:200])   # случайный текст - это нормально до обучения

In [ ]:
# Инициализируем нашу модель и посморим на ее архитектруру


model = BertSummarizer(bert_model_name=model_name)
model = model.to(device)
model

In [ ]:
# Посмотрим на генерацию без обучения

eval_data_sample = next(iter(eval_dataloader))
model.generate(
    eval_data_sample["input_ids"][:1].to(device),
    eval_data_sample["attention_mask"][:1].to(device),
    tokenizer,
)

'угле ##стеров ##ряз ##drop мнения прямои ##ru распах связа опове ##рогом необходимость подслу уходила коробка смарт шпион ##оцени реитинг учебник подош проидет 📝 правом англииском известные body подумали регла швеицар ##% ##ить шпион ##оцени реитинг помимо club12 ##ольше ##гант очертания ##лли ##зали собравшихся пошу группо мощныи хостинг удивлением настоящии υ'

## Обучение модели (1 балл)

0.25 балла за простейший рабочий цикл; 

0.5 балла за графики для лосса и метрик на трейне и валидации.

0.25 балла за логгинг в тензорборд или WandB

В данном разделе вам необходимо **реализовать цикл для обучения модели**


In [ ]:
# Пример обучения на одной итерации
# Все помнят, что надо предсказывать следующий токен?


def train_step(
    model, input_ids, attention_mask, decoder_input_ids, optimizer, criterion
):
    model.train()
    optimizer.zero_grad()
    outputs = model(input_ids, attention_mask, decoder_input_ids)
    loss = criterion(outputs.view(-1, outputs.size(-1)), decoder_input_ids.view(-1))
    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
# проверка одного шага перед запуском полного обучения
model.train()

batch = next(iter(train_dataloader))
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

decoder_input_ids = batch["labels"].clone()
decoder_input_ids[decoder_input_ids == -100] = tokenizer.pad_token_id
decoder_input_ids = decoder_input_ids.to(device)

# проверяем forward
with torch.no_grad():
    outputs = model(input_ids, attention_mask, decoder_input_ids)
print("форма outputs:", outputs.shape)

# проверяем один шаг обучения
criterion_check = nn.NLLLoss(ignore_index=tokenizer.pad_token_id)
optimizer_check = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_val = train_step(model, input_ids, attention_mask, decoder_input_ids, optimizer_check, criterion_check)
print("loss на одном батче:", round(loss_val, 4))

In [ ]:
# mini overfit проверка - убеждаемся, что модель вообще умеет учиться
# запускается один раз перед полным обучением, основной цикл не трогаем

from torch.utils.data import DataLoader, Subset

# берём 32 примера из train
mini_subset = Subset(tokenized_dataset, list(range(32)))
mini_loader = DataLoader(mini_subset, batch_size=8, shuffle=True)

mini_optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
mini_criterion = nn.NLLLoss(ignore_index=tokenizer.pad_token_id)

mini_losses = []

for epoch in range(30):
    epoch_loss = 0.0
    for batch in mini_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        decoder_input_ids = batch["labels"].clone()
        decoder_input_ids[decoder_input_ids == -100] = tokenizer.pad_token_id
        decoder_input_ids = decoder_input_ids.to(device)

        loss = train_step(model, input_ids, attention_mask, decoder_input_ids, mini_optimizer, mini_criterion)
        epoch_loss += loss

    avg = epoch_loss / len(mini_loader)
    mini_losses.append(avg)

    if (epoch + 1) % 5 == 0:
        print(f"эпоха {epoch + 1:2d}: loss = {avg:.4f}")

# график
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(mini_losses)
plt.xlabel("эпоха")
plt.ylabel("loss")
plt.title("mini overfit — loss должен падать вниз")
plt.tight_layout()
plt.show()

print("начальный loss:", round(mini_losses[0], 4))
print("финальный loss:", round(mini_losses[-1], 4))

In [ ]:
from tqdm import tqdm

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# NLLLoss совместим с LogSoftmax из forward; ignore_index убирает паддинг из расчёта ошибки
criterion = nn.NLLLoss(ignore_index=tokenizer.pad_token_id)

num_epochs = 3
train_losses = []
val_losses = []
best_val_loss = float("inf")

for epoch in range(num_epochs):
    model.train()
    epoch_train_loss = 0.0

    for batch in tqdm(train_dataloader, desc=f"эпоха {epoch + 1} / {num_epochs} — обучение"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # -100 нельзя подавать в embedding - заменяем на pad_token_id
        decoder_input_ids = batch["labels"].clone()
        decoder_input_ids[decoder_input_ids == -100] = tokenizer.pad_token_id
        decoder_input_ids = decoder_input_ids.to(device)

        loss = train_step(model, input_ids, attention_mask, decoder_input_ids, optimizer, criterion)
        epoch_train_loss += loss

    avg_train_loss = epoch_train_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)

    # валидация
    model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc=f"эпоха {epoch + 1} / {num_epochs} — валидация"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            decoder_input_ids = batch["labels"].clone()
            decoder_input_ids[decoder_input_ids == -100] = tokenizer.pad_token_id
            decoder_input_ids = decoder_input_ids.to(device)

            outputs = model(input_ids, attention_mask, decoder_input_ids)
            loss = criterion(outputs.view(-1, outputs.size(-1)), decoder_input_ids.view(-1))
            epoch_val_loss += loss.item()

    avg_val_loss = epoch_val_loss / len(eval_dataloader)
    val_losses.append(avg_val_loss)

    print(f"эпоха {epoch + 1}: train_loss = {avg_train_loss:.4f}, val_loss = {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_model.pt")
        tokenizer.save_pretrained("best_tokenizer")
        print("  -> лучшая модель сохранена")

print("обучение завершено")

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_losses, label="train loss")
plt.plot(epochs, val_losses, label="val loss")
plt.xlabel("эпоха")
plt.ylabel("loss")
plt.title("Кривые обучения")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import os

print("train losses по эпохам:", [round(l, 4) for l in train_losses])
print("val losses по эпохам:  ", [round(l, 4) for l in val_losses])

# проверяем, что loss не nan и не inf
for i, (tl, vl) in enumerate(zip(train_losses, val_losses)):
    if tl != tl or vl != vl:  # nan != nan всегда True
        print(f"ВНИМАНИЕ: nan в эпохе {i + 1}")
    if tl == float("inf") or vl == float("inf"):
        print(f"ВНИМАНИЕ: inf в эпохе {i + 1}")

# проверяем, что модель сохранилась
print("best_model.pt сохранён:", os.path.exists("best_model.pt"))

## Метрики качества (1 балл)

По 0.33 балла за измерение каждой из предлагаемых метрик

**Реализуйте функицию для подсчета метрик качества суммаризации.**

Что мы хотим считать:
 1. [HuggingFace Rouge](https://huggingface.co/spaces/evaluate-metric/rouge)
 2. [HuggingFace Bleu](https://huggingface.co/spaces/evaluate-metric/bleu)
 3. [HuggingFace BERT Score](https://huggingface.co/spaces/evaluate-metric/bertscore)

In [ ]:
def compute_metrics():
    # <YOUR CODE HERE>
    pass


def evaluation():
    # <YOUR CODE HERE>
    pass

## Обучение модели (0.5 балла)
**Обучите модель, сохраните лучшую версию** (метод `.save_pretrained()` объекта класса AutoModel... или `torch.save()`) **и добавьте пример генерации**. Учтите, что если изменялся токенизатор (а лучше просто по умолчанию), его тоже нужно сохранить.

Для сравнения оценки качества генерации по значениям реализованных метрик можете запустить ruT5-small без дообучения. Мы намеренно даем бейзлайн именно в таком виде.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("YOUR MODEL")
summary = #<YOUR CODE HERE>

## Реализация менее жадных стратегий выбора следующего токена (4 балла)
Всегда ли выбор наиболее вероятного токена на каждом шаге – это лучшая стратегия для генерации текста?

<details>
    <summary>Спойлер</summary>
    <p>Нет</p>
</details>

**Сравнение стратегий для генерации текста:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br><br/> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br><br/> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words <br/> that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br><br/> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps <br/> the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br><br/> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br><br/> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities <br/> by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br><br/> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br><br/> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br><br/> **Cons:** Computationally more complex and requires a good loss function. |

Ссылки на докуметацию:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

**1. Реализуйте стратегию Top-k в методе `generate`** (1 балл).   

**2. Реализуйте стратегию Nucleus Sampling (Top-p) в методе `generate`** (1 балл)

**3. Реализуйте стратегию Beam Search** (2 балла)

Получилось ли улучшить генерацию?

## Бонус (1 балл)

Что требуется сделать:

- от имеющейся модели "откусить" только декодерную часть
- написать цикл обучения (скорее поправить имеющийся) и дообучить декодер
- посмотреть качество генерации по метрикам и "глазами"
- ответить на вопрос "Дает ли применение Encoder-Decoder архитектуры значительный буст в качестве генерации?" с пруфами